# SAM paper replication: WideResNet-28-10 on CIFAR-10

## 1. Target and experimental contract
Replicate **Table 1, first row** of Foret et al., *Sharpness-Aware Minimization for Efficiently Improving Generalization* (ICLR 2021), [paper v3](https://arxiv.org/abs/2010.01412v3). The central empirical finding is lower test error with SAM. This notebook does not test whether flatness causes that improvement.

| Setting | Planned replication |
|---|---|
| Model/data | Authors' WideResNet-28-10; full 50,000 train / 10,000 official test images |
| Runs | SAM-200, SGD-200, separately trained SGD-400; seeds 0, 1, 2 |
| Batch | Global 256; eight logical sub-batches of 32 on one A100 |
| Optimizer | Authors' Nesterov momentum 0.9, LR 0.1, full-duration cosine, WD 0.0005, clipping 5 |
| Augmentation | Authors' basic flip, reflection padding and crop; no cutout/AutoAugment |
| SAM | rho 0.05; independent sub-batch perturbations, then averaged gradients |
| Paper reference | SAM **2.7 +/- 0.1%**, SGD **3.5 +/- 0.1%** test error; mean and 95% CI over five replicas |

We use three seeds ([course minimum](https://spkreddy.org/csci567project.html)) instead of five, and one A100 instead of eight V100s. Named-axis vectorization reuses the original SAM step, keeping eight BatchNorm histories. The adaptation requires preflight verification; it is not a bitwise reproduction of eight physical GPUs. Hyperparameters are fixed from Appendix C.1, without test-set tuning. FP32; 195 full training batches per epoch (80 shuffled examples dropped, as in the released pipeline).

The paper allows SGD twice as many epochs. We report both separately trained durations and the per-seed lower final test error across them, labeled as paper-style selection. Never select an intermediate checkpoint by test performance. Earlier ResNet-18 pilots remain separate feasibility experiments.

**Nothing runs by default.** Enable switches only when ready. No results or successful A100 validation are claimed in this unexecuted notebook.

**Predeclared reproduction rule (fixed before training):** Assess only after all nine planned runs finish. For each seed define paper-style SGD as the lower final error of its independent SGD-200 and SGD-400 runs. Call this row **reproduced** only if (1) mean SAM error is lower than mean paper-style SGD error, (2) the two-sided paired Student-t 95% CI for `paper-style SGD minus SAM` is entirely above zero, and (3) the SAM and paper-style SGD means are within **0.3 percentage points inclusive** of 2.7% and 3.5%, respectively. A CI touching zero fails the rule. Otherwise report **criterion not met**, not proof that SAM has no effect. Until all runs finish: **not assessed (to verify)**. This is our operational rule, not a criterion specified by the paper.

Also report same-seed differences `SGD-200 minus SAM` and `SGD-400 minus SAM`, alongside the paper-style difference, with sample mean, sample SD (ddof=1) and two-sided t-based 95% CI (df=n-1). Positive differences favor SAM; no unpaired error-bar subtraction. These secondary comparisons do not replace the predeclared primary paper-style comparison.

**Resource contract:** 40 cumulative worker GPU-hours including preflight and retries; smoke time is included when using/transferring the same authoritative ledger. Runtime behavior and feasibility of completing the full nine-run matrix are **(to verify)**.


Runtime note: `CONFIG['deterministic_reductions']` defaults to `False`. The Jetstream `--deterministic-reductions` option explicitly enables the GPU reduction rewrite supported by the pinned XLA version. This is separate from cuDNN determinism. Both settings are recorded and must be retained across preflight/training/resume; no tolerances or original optimizer logic are changed. A100 resolution of the observed reference-repeatability failure remains **(to verify)**.


## 2. Environment and configuration
Use Jupyter on the course A100 with persistent storage. Set paths below first; `WORK_ROOT` must be outside the source checkout and survive job termination. The kernel may use modern Python; workers use the explicit Python 3.8 interpreter.

Optional setup downloads a standalone micromamba into WORK_ROOT/bin when needed; it does not restart the kernel. It clones a pinned fork revision into a folder named `sam` for the original imports. Original requirements stay unchanged: a runtime copy applies the previously tested absl-py compatibility adjustment. CUDA compatibility on the course A100 remains to verify. Failed setup stops; there is no automatic JAX upgrade or replacement implementation.

Rerunning configuration to enable a switch keeps the current SESSION. Use START_NEW_SESSION=True once to intentionally create a new session, then return it to False. After a kernel restart, set REUSE_SESSION explicitly to recover the same folder.


In [ ]:
from pathlib import Path
from datetime import datetime, timezone
import os, sys, json, hashlib, subprocess, shutil, uuid, ast, signal, time

WORK_ROOT = Path.home() / 'sam_paper_replication'  # Use your persistent allocation storage.
ENV_PREFIX = Path.home() / '.conda' / 'envs' / 'sam'
PYTHON = ENV_PREFIX / 'bin' / 'python'
REPO = WORK_ROOT / 'work' / 'sam'
REVISION = 'f7b9ff46ee0deab8004d5508baf68b7e7c3dad1a'
DATA_DIR = WORK_ROOT / 'tensorflow_datasets'  # Or point to a complete existing TFDS cache.
REUSE_SESSION = None  # Absolute path to the original session when explicitly resuming.
START_NEW_SESSION = False  # Set True once to intentionally start a new experiment.
if REUSE_SESSION:
    SESSION = Path(REUSE_SESSION)
elif START_NEW_SESSION or 'SESSION' not in globals():
    SESSION = WORK_ROOT / 'runs' / (datetime.now(timezone.utc).strftime('%Y%m%d_%H%M%S') + '_' + uuid.uuid4().hex[:6])
SEEDS = [0, 1, 2]
RUN_SETUP = False
RUN_PREFLIGHT = False
RUN_TRAINING = False
RUN_DIAGNOSTIC_SMOKE = False
RENDER_RESULTS = False
REPLICATION_BUDGET_HOURS = 40.0  # Fixed cumulative ceiling, including preflight and retries.
BUDGET_FILE = WORK_ROOT / 'gpu_budget.json'  # Shared across sessions; never reset to regain time.
RUN_SELECTION = 'SAM-200_seed_0'  # One run, or explicitly 'all' for the nine-run matrix.
RESUME_EXISTING = False
CONFIG = dict(mode='replication',deterministic_reductions=False,tf_deterministic_ops=os.environ.get('TF_DETERMINISTIC_OPS','0'),cudnn_deterministic=os.environ.get('TF_CUDNN_DETERMINISTIC','0'),budget_limit_hours=40.0,budget_file=str(BUDGET_FILE),revision=REVISION, repo=str(REPO), session=str(SESSION), data_dir=str(DATA_DIR),
              seeds=SEEDS, logical_replicas=8, batch_size=256, subbatch_size=32,
              lr=0.1, weight_decay=0.0005, rho=0.05, gradient_clipping=5.0)
def configured_xla_flags():
    # Supported by the XLA commit bundled with jaxlib 0.1.57; not a modern-only flag.
    return ('--xla_gpu_cuda_data_dir='+str(WORK_ROOT/'cuda')+
            ' --xla_gpu_deterministic_reductions='+str(CONFIG['deterministic_reductions']).lower())

print('Planned session:', SESSION)
print('Setup, GPU execution and result rendering default to disabled.')


In [ ]:
import traceback, urllib.request, tarfile, io, platform

def setup_command(command, environment=None):
    print('Setup:', ' '.join(map(str,command[:6])),flush=True)
    with (SESSION/'setup.log').open('a') as log:
        log.write('\nCOMMAND: '+repr(command)+'\n');log.flush()
        subprocess.run(list(map(str,command)),stdout=log,stderr=subprocess.STDOUT,check=True,timeout=1800,env=environment)
    print('\n'.join((SESSION/'setup.log').read_text(errors='replace').splitlines()[-5:]),flush=True)

def cuda_environment():
    environment=os.environ.copy()
    environment['PATH']=str(ENV_PREFIX/'bin')+os.pathsep+environment.get('PATH','')
    environment['LD_LIBRARY_PATH']=str(ENV_PREFIX/'lib')+os.pathsep+environment.get('LD_LIBRARY_PATH','')
    environment['XLA_FLAGS']=configured_xla_flags()
    candidates=[ENV_PREFIX/'bin'/'ptxas']+sorted(Path('/usr/local').glob('cuda-11*/bin/ptxas'))
    for candidate in candidates:
        if not candidate.is_file():
            continue
        result=subprocess.run([str(candidate),'--version'],env=environment,text=True,capture_output=True,timeout=30)
        if result.returncode==0 and 'release 11.' in (result.stdout+result.stderr):
            toolkit=candidate.parent.parent
            environment['CUDA_HOME']=str(toolkit)
            environment['CUDA_PATH']=str(toolkit)
            environment['PATH']=str(candidate.parent)+os.pathsep+environment['PATH']
            return environment
    raise RuntimeError('No working CUDA 11.x ptxas found; checked the pinned environment and /usr/local/cuda-11*/bin.')


if RUN_SETUP:
    WORK_ROOT.mkdir(parents=True,exist_ok=True)
    SESSION.mkdir(parents=True,exist_ok=True)
    SETUP_OK=False
    FAILURES=[]
    try:
        assert RUN_SETUP and not RUN_TRAINING
        assert sys.platform=='linux' and platform.machine()=='x86_64', 'Use a Linux x86-64 Colab GPU runtime.'
        setup_command(['nvidia-smi','-L'])
        manager=WORK_ROOT/'bin'/'micromamba'
        if not manager.exists():
            print('Downloading standalone environment manager (no kernel restart).',flush=True)
            with urllib.request.urlopen('https://micro.mamba.pm/api/micromamba/linux-64/latest',timeout=120) as response:
                data=response.read()
            with tarfile.open(fileobj=io.BytesIO(data),mode='r:bz2') as archive:
                member=archive.getmember('bin/micromamba')
                assert member.isfile()
                manager.parent.mkdir(parents=True,exist_ok=True)
                with archive.extractfile(member) as executable:
                    manager.write_bytes(executable.read())
            manager.chmod(0o755)
            (SESSION/'bootstrap_sha256.txt').write_text(hashlib.sha256(manager.read_bytes()).hexdigest())
        setup_command([manager,'--version'])
        os.environ['MAMBA_ROOT_PREFIX']=str(WORK_ROOT/'mamba')
        REPO.parent.mkdir(parents=True,exist_ok=True)
        if not REPO.exists():
            setup_command(['git','clone','https://github.com/Siddharth-magesh/sam-replication.git',REPO])
            setup_command(['git','-C',REPO,'checkout','--detach',REVISION])
        assert subprocess.check_output(['git','-C',str(REPO),'rev-parse','HEAD'],text=True).strip()==REVISION
        action='install' if PYTHON.exists() else 'create'
        setup_command([manager,action,'-p',ENV_PREFIX,'-c','conda-forge','--override-channels',
                       'python=3.8','pip=24.0','cudatoolkit=11.0','cudnn=8.0.5','-y'])
        setup_command([manager,'install','-p',ENV_PREFIX,'-c','nvidia/label/cuda-11.3.1','--override-channels','cuda-nvcc=11.3.122','-y'])
        requirements=[s.replace('absl-py==0.11.0','absl-py==0.10.0') for s in
                      (REPO/'requirements.txt').read_text().splitlines() if not s.startswith('jaxlib')]
        fixed=WORK_ROOT/'requirements_runtime.txt'
        fixed.write_text('\n'.join(requirements)+'\n')
        setup_command([PYTHON,'-m','pip','install','jaxlib==0.1.57+cuda110','-f',
                       'https://storage.googleapis.com/jax-releases/jax_cuda_releases.html'])
        setup_command([PYTHON,'-m','pip','install','-r',fixed])
        candidates=list(ENV_PREFIX.rglob('libdevice*.bc'))
        assert candidates,'Pinned CUDA libdevice missing.'
        libdir=WORK_ROOT/'cuda'/'nvvm'/'libdevice';libdir.mkdir(parents=True,exist_ok=True)
        shutil.copy2(candidates[0],libdir/'libdevice.10.bc')
        environment=cuda_environment()
        ptxas=shutil.which('ptxas',path=environment['PATH'])
        assert ptxas,'CUDA 11.0 ptxas is missing from the pinned environment.'
        ptxas_result=subprocess.run([ptxas,'--version'],env=environment,text=True,capture_output=True,timeout=30)
        assert ptxas_result.returncode==0,'ptxas --version failed: '+ptxas_result.stderr
        print('Using ptxas:',ptxas,ptxas_result.stdout.strip(),flush=True)
        setup_command([PYTHON,'-c','import jax; x=jax.jit(lambda value: value @ value)(jax.numpy.ones((2,2))); x.block_until_ready(); assert any(device.platform=="gpu" for device in jax.devices()); print(jax.devices(),x)'],environment=environment)
        setup_command([PYTHON,'-c','from importlib.metadata import version; print({p:version(p) for p in ["numpy","tensorflow","jax","jaxlib","flax"]})'])
        SETUP_OK=True
        print('Pinned setup complete. Enable preflight when ready.')
    except Exception as error:
        FAILURES.append(dict(stage='setup',error=str(error),traceback=traceback.format_exc()))
        print('SETUP FAILED; no GPU work will launch. Inspect SESSION/setup.log.')
        print(str(error))
    assert SETUP_OK, 'Pinned setup failed; inspect SESSION/setup.log.'
else:
    print('Setup disabled. Configure ENV_PREFIX to reuse an existing pinned environment.')


## 3. Preflight and runtime estimates
Both workers import the authors' original model, optimizer, `train_step` and `eval_step`. Only execution orchestration changes. The vectorized step has an unbatched optimizer and learning rate (`out_axes=None`) and eight batched BN states/metrics. This removes redundant optimizer replication and avoids mistakenly slicing a shared parameter dimension. The original gradient averaging, clipping and SAM perturbations are unchanged.

Preflight compares one logical replica with the original `pmap` step; checks parameter updates and finite output; compares every retained BN lane against its clean first-pass forward; checks independent BN histories and mean-state evaluation; evaluates all 10,000 test examples; and round-trips a checkpoint. Fixed tolerances are never relaxed automatically. These checks are written but their GPU outcomes are **(to verify)**.

Then one full epoch per method is timed with the 200-epoch schedule. Estimates use actual synchronized step durations (including input delivery), p90 steady-step time, full evaluation, checkpoint I/O, and measured initialization/diagnostic compilation overhead, plus 20% planning allowance. The same startup allowance is conservative for SGD-400. Resumed estimates subtract only committed epochs. All preflight time is charged to the same 40-hour ledger; it does not supply final research results.

**Current readiness:** the T4 functional SGD smoke passed; the eight-replica SAM smoke exhausted T4 memory. The original GPU step also failed its own repeatability tolerance. A100 memory capacity and strict numerical preflight remain **(to verify)**. Start with setup and preflight only; do not bypass a failed numerical or budget gate. Transfer the authoritative GPU ledger before the first launch.


In [ ]:
WORKER_SOURCE = r'''# Orchestration around the authors' unchanged SAM implementation (Apache-2.0).
import os, sys, json, time, hashlib, traceback, subprocess, platform, shutil
from pathlib import Path
from functools import partial
phase, config_path, label, seed_text, resume_text = sys.argv[1:]
C = json.loads(Path(config_path).read_text())
seed, resume = int(seed_text), resume_text == 'true'
EPOCHS = {'SAM-200':200, 'SGD-200':200, 'SGD-400':400}
assert phase in ('preflight','train','smoke') and label in EPOCHS
assert C['mode'] in ('replication','smoke')
assert (phase=='smoke') == (C['mode']=='smoke')
assert phase!='smoke' or (label in ('SAM-200','SGD-200') and seed==0)
# A worker must be started by the persistent budget supervisor.
if 'SAM_STOP_MONOTONIC' not in os.environ:
    raise RuntimeError('Launch through the budget supervisor; direct worker execution is disabled.')
def budget_check():
    if time.monotonic() >= float(os.environ['SAM_STOP_MONOTONIC']):
        raise TimeoutError('Cumulative GPU budget reached; last committed checkpoint is preserved.')
budget_check()
run = Path(C['session'])/phase/(label+'_seed_'+str(seed))
run.mkdir(parents=True, exist_ok=True)
started, prior_seconds = time.monotonic(), 0.0
active_run = False

def write_json(path, value):
    path = Path(path); temporary = path.with_name(path.name+'.tmp')
    with temporary.open('w') as stream:
        stream.write(json.dumps(value, indent=2, allow_nan=False)+'\n'); stream.flush(); os.fsync(stream.fileno())
    os.replace(str(temporary), str(path))

def check(condition, message):
    if not condition:
        raise RuntimeError(message)

def run_experiment():
    global prior_seconds, active_run
    check(os.environ.get('TF_CUDNN_DETERMINISTIC','0')==C['cudnn_deterministic'],
          'cuDNN determinism setting differs from the saved run configuration.')
    print('Recorded TF_CUDNN_DETERMINISTIC='+C['cudnn_deterministic'],flush=True)
    check(os.environ.get('TF_DETERMINISTIC_OPS','0')==C['tf_deterministic_ops'],
          'TF_DETERMINISTIC_OPS differs from saved configuration.')
    reduction_flags=[v for v in os.environ.get('XLA_FLAGS','').split()
                     if v.startswith('--xla_gpu_deterministic_reductions')]
    check(type(C['deterministic_reductions']) is bool, 'Reduction setting must be boolean.')
    check(reduction_flags==['--xla_gpu_deterministic_reductions='+str(C['deterministic_reductions']).lower()],
          'XLA reduction setting differs from saved configuration.')
    runtime_controls=dict(xla_flags=os.environ.get('XLA_FLAGS',''),
        tf_cudnn_deterministic=C['cudnn_deterministic'],tf_deterministic_ops=C['tf_deterministic_ops'])
    print('Recorded XLA_FLAGS='+runtime_controls['xla_flags'],flush=True)
    write_json(run/'runtime_controls.json',runtime_controls)
    import numpy as np
    import tensorflow as tf
    tf.config.experimental.set_visible_devices([], 'GPU')
    import jax, jaxlib, flax
    import jax.numpy as jnp
    from flax import serialization
    from absl import flags
    sys.path.insert(0, str(Path(C['repo']).parent))
    from sam.sam_jax.models import load_model, wide_resnet
    from sam.sam_jax.datasets import dataset_source
    from sam.sam_jax.training_utils import flax_training as official
    FLAGS = flags.FLAGS
    FLAGS(['replication'])
    settings = dict(learning_rate=0.1, weight_decay=0.0005, gradient_clipping=5.0,
        use_rmsprop=False, sam_rho=0.05 if label=='SAM-200' else 0.0,
        sync_perturbations=False, label_smoothing=0.0, no_weight_decay_on_bn=False,
        ema_decay=0.0, use_test_set=True, use_additional_skip_connections=False,
        run_seed=seed, compute_top_5_error_rate=False)
    for key,value in settings.items():
        setattr(FLAGS,key,value)
    versions = dict(python=platform.python_version(), jax=jax.__version__,
        jaxlib=jaxlib.__version__, flax=flax.__version__, tensorflow=tf.__version__, numpy=np.__version__)
    check(sys.version_info[:2]==(3,8), 'Use the pinned Python 3.8 interpreter.')
    check(versions['jax']=='0.2.6' and versions['flax']=='0.2.2', 'JAX/Flax mismatch.')
    check(versions['jaxlib'].split('+')[0]=='0.1.57', 'Expected jaxlib 0.1.57; CUDA execution is checked next.')
    devices = jax.local_devices()
    check(len(devices)==1 and devices[0].platform=='gpu', 'Expose exactly one GPU.')
    gpu = subprocess.check_output(['nvidia-smi','--query-gpu=name,uuid,memory.total','--format=csv,noheader'],text=True).strip()
    def gpu_memory():
        value=subprocess.check_output(['nvidia-smi','--query-gpu=memory.used','--format=csv,noheader,nounits'],text=True).strip()
        check(value.isdigit(),'GPU memory query returned a non-numeric value.')
        return int(value)
    # Query the CUDA-visible ordinal, not all GPUs reported by nvidia-smi.
    import ctypes
    cuda = ctypes.CDLL('libcuda.so.1')
    ordinal = ctypes.c_int(); name = ctypes.create_string_buffer(256)
    check(cuda.cuInit(0)==0 and cuda.cuDeviceGet(ctypes.byref(ordinal),0)==0, 'CUDA identity query failed.')
    check(cuda.cuDeviceGetName(name,256,ordinal)==0, 'CUDA device name unavailable.')
    selected_gpu = name.value.decode()
    if C['mode'] in ('replication','smoke'):
        check('A100' in selected_gpu, 'Main replication requires the selected GPU to be an A100.')
    check(versions['tensorflow']=='2.3.1' and versions['numpy']=='1.18.5', 'Legacy TensorFlow/NumPy pins changed.')
    revision = subprocess.check_output(['git','-C',C['repo'],'rev-parse','HEAD'],text=True).strip()
    check(revision==C['revision'], 'Repository revision mismatch.')
    check(not subprocess.check_output(['git','-C',C['repo'],'status','--porcelain','--untracked-files=no'],text=True).strip(), 'Tracked runtime source was modified.')
    source_hashes = {str(p.relative_to(C['repo'])):hashlib.sha256(p.read_bytes()).hexdigest()
        for p in sorted(Path(C['repo']).rglob('*.py')) if '.git' not in p.parts}
    check(C['seeds']==[0,1,2] and C['batch_size']==256 and C['subbatch_size']==32 and C['logical_replicas']==8
          and C['lr']==0.1 and C['weight_decay']==0.0005 and C['rho']==0.05 and C['gradient_clipping']==5.0,
          'This notebook implements the fixed paper contract; config changes require a reviewed new experiment.')
    identity = dict(config=C, label=label, seed=seed, settings=settings, versions=versions,
        source_hashes=source_hashes, worker_sha256=hashlib.sha256(Path(__file__).read_bytes()).hexdigest())
    signature = hashlib.sha256(json.dumps(identity,sort_keys=True).encode()).hexdigest()
    if phase == 'train':
        for method in ('SAM-200','SGD-200'):
            pf = Path(C['session'])/'preflight'/(method+'_seed_0')
            report = json.loads((pf/'preflight.json').read_text())
            prior_manifest = json.loads((pf/'manifest.json').read_text())
            check(report['status']=='passed' and json.loads((pf/'status.json').read_text())['status']=='complete', 'Preflight is incomplete.')
            check(report['signature']==prior_manifest['signature'], 'Preflight provenance mismatch.')
            for key in ('config','versions','source_hashes','worker_sha256'):
                check(prior_manifest['identity'][key]==identity[key], 'Environment/source/config changed since preflight: '+key)
    manifest_path = run/'manifest.json'
    if manifest_path.exists():
        check(json.loads(manifest_path.read_text())['signature']==signature, 'Code/config/environment changed: use a new session.')
        old = json.loads((run/'status.json').read_text()) if (run/'status.json').exists() else {}
        prior_seconds = float(old.get('allocated_wall_seconds',0))
        if old.get('status')=='complete':
            print('Already complete:',run,flush=True)
            return
        check(resume and phase=='train', 'Incomplete existing run: explicitly resume or use a new session.')
    else:
        check(not resume, 'Cannot resume a run without a manifest.')
        write_json(manifest_path,dict(signature=signature,identity=identity))
    active_run = True
    write_json(run/'environment.json',dict(versions=versions,gpu=gpu,selected_gpu=selected_gpu,revision=revision,logical_replicas=8,sam_subbatch=32))
    write_json(run/'status.json',dict(status='running',allocated_wall_seconds=prior_seconds))
    np.random.seed(seed); tf.random.set_seed(seed)
    ds = dataset_source.Cifar10(256,'basic','none')
    check(ds.num_training_obs==50000, 'Expected 50,000 training examples.')
    module = wide_resnet.WideResnet.partial(blocks_per_group=4,channel_multiplier=10,num_outputs=10)
    # Explicit seed bypasses get_model's fixed default and PRNG-array truth test.
    model, initial_state = load_model.create_image_model(jax.random.PRNGKey(seed),32,32,module,3)
    parameter_count = sum(int(x.size) for x in jax.tree_leaves(model.params))
    write_json(run/'architecture_check.json', dict(parameters=parameter_count,
        architecture='WideResnet28x10', matched=True,
        blocks_per_group=4, channel_multiplier=10, num_outputs=10))
    optimizer = official.create_optimizer(model,0.0)
    initialization = dict(seed=seed,
        parameter_sha256=hashlib.sha256(serialization.to_bytes(model.params)).hexdigest(),
        parameters=sum(int(x.size) for x in jax.tree_leaves(model.params)))
    if (run/'initialization.json').exists():
        check(json.loads((run/'initialization.json').read_text()) == initialization,
              'Seeded initialization changed since this run started.')
    else:
        write_json(run/'initialization.json', initialization)
    states = jax.tree_map(lambda x:jnp.stack([x]*8),initial_state)
    schedule = official.get_cosine_schedule(EPOCHS[label],0.1,50000,256)
    original_step = partial(official.train_step,learning_rate_fn=schedule,l2_reg=0.0005)
    # Original pmean averages independent gradients BEFORE clipping and momentum.
    # pmean makes optimizer and LR unmapped. None avoids eight full optimizer copies.
    mapped_step = jax.jit(jax.vmap(original_step,in_axes=(None,0,0,0),
                                  out_axes=(None,0,0,None),axis_name='batch'))
    mapped_eval = jax.jit(jax.vmap(official.eval_step,in_axes=(None,0,0),axis_name='batch'))

    def synchronize(tree):
        for x in jax.tree_leaves(tree):
            if hasattr(x,'block_until_ready'):
                x.block_until_ready()
        return tree

    def finite(tree):
        return all(np.isfinite(np.asarray(x)).all() for x in jax.tree_leaves(tree))

    def keys_for(step,count):
        return jax.random.split(jax.random.fold_in(jax.random.PRNGKey(seed),int(step)),count)

    def train_batches(epoch):
        # Stateful TF augmentation/GPU kernels need not be bitwise deterministic.
        tf.random.set_seed(seed*100000+epoch)
        options = tf.data.Options(); options.experimental_deterministic=True
        return ds.get_train(True).with_options(options)

    def shard(batch,count=8):
        return {k:np.asarray(v).reshape((count,-1)+tuple(v.shape[1:])) for k,v in batch.items()}

    def differences(a,b):
        check(jax.tree_structure(a)==jax.tree_structure(b), 'Compared state structures differ.')
        max_abs,sqdiff,sqref,first_divergent_leaf = 0.0,0.0,0.0,None
        for leaf_index,(x,y) in enumerate(zip(jax.tree_leaves(a),jax.tree_leaves(b))):
            check(np.shape(x)==np.shape(y), 'Compared array shapes differ.')
            x,y = np.asarray(x,dtype=np.float64),np.asarray(y,dtype=np.float64)
            leaf_max_abs=float(np.max(np.abs(x-y)))
            if leaf_max_abs>0 and first_divergent_leaf is None:
                first_divergent_leaf=leaf_index
            max_abs=max(max_abs,leaf_max_abs)
            sqdiff+=float(np.sum((x-y)**2)); sqref+=float(np.sum(y**2))
        return dict(max_abs=max_abs,relative_l2=(sqdiff**0.5)/max(sqref**0.5,1e-12),
                    first_divergent_leaf=first_divergent_leaf)

    def eval_full(opt,bn,limit_batches=None):
        sums=dict(error_rate=0.0,loss=0.0); seen=0
        for batch_index,batch in enumerate(ds.get_test()):
            if limit_batches is not None and batch_index>=limit_batches:
                break
            budget_check()
            raw={k:np.asarray(v) for k,v in batch.items()}
            n=len(raw['label']); padded=((n+7)//8)*8
            raw={k:np.pad(v,[(0,padded-n)]+[(0,0)]*(v.ndim-1)) for k,v in raw.items()}
            raw['mask']=np.concatenate([np.ones(n),np.zeros(padded-n)]).astype(np.float32)
            out=synchronize(mapped_eval(opt.target,bn,shard(raw)))
            check(finite(out),'Non-finite evaluation.')
            for key in sums:
                sums[key]+=float(np.asarray(out[key])[0])
            seen+=n
        check(seen==(10000 if limit_batches is None else limit_batches*32),'Evaluation sample count mismatch.')
        return dict(test_error_pct=100*sums['error_rate']/seen,
            test_accuracy_pct=100*(1-sums['error_rate']/seen),test_loss=sums['loss']/seen,test_examples=seen)

    history=[]; completed=0
    checkpoints=run/'checkpoints'; checkpoints.mkdir(exist_ok=True)

    def save_checkpoint(epoch,opt,bn,rows):
        folder=checkpoints/('epoch_%04d'%epoch)
        check(not folder.exists(),'Checkpoint exists; refusing overwrite.')
        temporary=checkpoints/('writing_%04d'%epoch); temporary.mkdir(exist_ok=True)
        payload=dict(optimizer=opt,states=bn,completed_epoch=int(epoch))
        with (temporary/'state.msgpack').open('wb') as stream:
            stream.write(serialization.to_bytes(payload)); stream.flush(); os.fsync(stream.fileno())
        write_json(temporary/'history.json',rows)
        write_json(temporary/'metadata.json',dict(signature=signature,completed_epoch=epoch,checkpoint_unit='epoch',
            state_sha256=hashlib.sha256((temporary/'state.msgpack').read_bytes()).hexdigest()))
        os.replace(str(temporary),str(folder))
        write_json(checkpoints/'latest.json',dict(folder=folder.name))
        for old in sorted(checkpoints.glob('epoch_*'))[:-2]:
            check(old.resolve().parent==checkpoints.resolve(),'Unsafe checkpoint cleanup.')
            shutil.rmtree(str(old))
        return folder

    def load_checkpoint():
        committed = sorted(checkpoints.glob('epoch_*'))
        check(bool(committed), 'No complete epoch checkpoint; start a fresh run/session.')
        # Recover even if killed after directory commit but before latest.json update.
        folder=committed[-1]
        check(folder.resolve().parent==checkpoints.resolve(),'Invalid checkpoint pointer.')
        meta=json.loads((folder/'metadata.json').read_text()); data=(folder/'state.msgpack').read_bytes()
        check(meta['signature']==signature and hashlib.sha256(data).hexdigest()==meta['state_sha256'],'Checkpoint provenance mismatch.')
        template=dict(optimizer=optimizer,states=states,completed_epoch=0)
        restored=serialization.from_bytes(template,data)
        rows=json.loads((folder/'history.json').read_text()); epoch=int(restored['completed_epoch'])
        check(epoch==meta['completed_epoch']==len(rows),'Checkpoint/history disagree.')
        check(int(restored['optimizer'].state.step)==epoch*195,'Checkpoint step mismatch.')
        return restored['optimizer'],restored['states'],epoch,rows

    if resume:
        optimizer,states,completed,history=load_checkpoint()
        write_json(run/'history.json',history)
    if phase == 'preflight':
        batch=next(iter(train_batches(0)))
        small={k:np.asarray(v)[:32][None,...] for k,v in batch.items()}
        one_state=jax.tree_map(lambda x:jnp.asarray(x)[None,...],initial_state); one_keys=keys_for(0,1)
        vmapped_one=jax.jit(jax.vmap(original_step,in_axes=(None,0,0,0),out_axes=(None,0,0,None),axis_name='batch'))
        pmapped_one=jax.pmap(original_step,axis_name='batch')
        def clone_arrays(tree):
            return jax.tree_map(lambda x:jnp.array(np.asarray(x)),tree)
        def clone_serialized(value):
            return serialization.from_bytes(value,serialization.to_bytes(value))
        reference_input=(clone_serialized(optimizer),clone_serialized(one_state),clone_arrays(small),clone_arrays(one_keys))
        repeated_input=(clone_serialized(optimizer),clone_serialized(one_state),clone_arrays(small),clone_arrays(one_keys))
        candidate_input=(clone_serialized(optimizer),clone_serialized(one_state),clone_arrays(small),clone_arrays(one_keys))
        input_identity=dict(optimizer_target=differences(reference_input[0].target,repeated_input[0].target),
                            optimizer_state=differences(reference_input[0].state,repeated_input[0].state),
                            batchnorm=differences(reference_input[1],repeated_input[1]),
                            inputs=differences(reference_input[2],repeated_input[2]),
                            rng=differences(reference_input[3],repeated_input[3]))
        check(all(value['max_abs']==0 for value in input_identity.values()),
              'One-replica diagnostic inputs differ before execution.')
        # Prepare independent transformations of the same original train_step.
        def pmap_arguments(inputs):
            # pmap maps axis 0 of EVERY optimizer leaf, including scalar step.
            opt,bn,images,keys=inputs
            replicated=jax.tree_map(lambda x:jnp.asarray(x)[None,...],opt)
            check(all(np.shape(y)==(1,)+np.shape(x) for x,y in
                      zip(jax.tree_leaves(opt),jax.tree_leaves(replicated))),
                  'Reference optimizer must have a leading one-device axis.')
            return replicated,bn,images,keys
        # Compile/warm both transforms before measurement. Reuse the exact same
        # prepared reference arguments, avoiding a separate reconstruction/transfer.
        prepared_reference=synchronize(pmap_arguments(reference_input))
        prepared_candidate=synchronize(candidate_input)
        def input_digest(inputs):
            digest=hashlib.sha256()
            for leaf in jax.tree_leaves(inputs):
                array=np.asarray(leaf)
                digest.update(str((array.dtype.str,array.shape)).encode())
                digest.update(array.tobytes())
            return digest.hexdigest()
        reference_input_hash=input_digest(prepared_reference)
        candidate_input_hash=input_digest(prepared_candidate)
        print('Warming original pmap and candidate vmap before numerical comparison.',flush=True)
        warm=synchronize(pmapped_one(*prepared_reference)); del warm
        warm=synchronize(vmapped_one(*prepared_candidate)); del warm
        reference=synchronize(pmapped_one(*prepared_reference))
        repeated=synchronize(pmapped_one(*prepared_reference))
        candidate=synchronize(vmapped_one(*prepared_candidate))
        input_integrity=dict(reference_unchanged=input_digest(prepared_reference)==reference_input_hash,
                             candidate_unchanged=input_digest(prepared_candidate)==candidate_input_hash)
        write_json(run/'reference_input_integrity.json',input_integrity)
        check(all(input_integrity.values()),'Numerical diagnostic inputs mutated during execution.')
        reference_optimizer=jax.tree_map(lambda x:x[0],reference[0])
        numerical = dict(parameters=differences(candidate[0].target,reference_optimizer.target),
                         optimizer_state=differences(candidate[0].state,reference_optimizer.state),
                         batchnorm=differences(candidate[1],reference[1]),
                         metrics=differences(candidate[2],reference[2]),
                         learning_rate=differences(candidate[3],reference[3][0]))
        checks=dict(one_replica_finite=finite(candidate) and finite(reference),
                    reference_protocol='warmed_same_prepared_inputs_v1',
                    runtime_controls=runtime_controls,
                    input_integrity=input_integrity,
                    input_identity=input_identity,one_replica_difference=numerical)
        write_json(run/'diagnostics.json',checks)
        repeated_optimizer=jax.tree_map(lambda x:x[0],repeated[0])
        repeat_difference=dict(parameters=differences(repeated_optimizer.target,reference_optimizer.target),
                               optimizer_state=differences(repeated_optimizer.state,reference_optimizer.state),
                               batchnorm=differences(repeated[1],reference[1]),
                               metrics=differences(repeated[2],reference[2]),
                               learning_rate=differences(repeated[3],reference[3]))
        strict_close=lambda ds:all(d['max_abs']<=1e-3 and d['relative_l2']<=1e-3 for d in ds.values())
        reference_repeatable=strict_close(repeat_difference)
        cross_transform_close=strict_close(numerical)
        noise_bound=all(numerical[k][metric]<=max(1e-3,3*repeat_difference[k][metric])
                       for k in numerical for metric in ('max_abs','relative_l2'))
        checks.update(reference_repeat_difference=repeat_difference,
                      reference_repeatable=reference_repeatable,
                      cross_transform_close=cross_transform_close,
                      within_reference_noise_bound=noise_bound,
                      numerical_equivalence=('passed' if reference_repeatable and cross_transform_close else
                          'inconclusive_original_not_repeatable' if not reference_repeatable and noise_bound else 'failed'))
        write_json(run/'diagnostics.json',checks)
        print('One-replica numerical comparison:',checks['numerical_equivalence'],flush=True)
        # This is the mandatory strict preflight gate, including original-kernel repeatability.
        acceptable=reference_repeatable and cross_transform_close
        check(checks['one_replica_finite'] and finite(repeated),
              'One-replica diagnostic produced non-finite output.')
        if C.get('diagnostic_only',False):
            write_json(run/'diagnostic_status.json',dict(status='complete',
                reference_repeatable=reference_repeatable,cross_transform_close=cross_transform_close,
                within_reference_noise_bound=noise_bound))
            write_json(run/'status.json',dict(status='complete',diagnostic_only=True,
                allocated_wall_seconds=prior_seconds+time.monotonic()-started))
            print('One-replica diagnostic complete; strict preflight not run.',flush=True)
            return
        check(acceptable,'One-replica numerical verification failed; inspect diagnostics.')
        del repeated,repeated_optimizer,reference,reference_optimizer,candidate,vmapped_one,pmapped_one,reference_input,repeated_input,candidate_input,prepared_reference,prepared_candidate
        # Free discarded comparison buffers before the full eight-lane step.
        import gc
        gc.collect()
        print('Checking eight logical replicas and BatchNorm.',flush=True)
        out=synchronize(mapped_step(optimizer,states,shard(batch),keys_for(0,8)))
        checks['eight_replica_finite']=finite(out)
        checks['shared_optimizer_step']=int(out[0].state.step)==int(optimizer.state.step)+1
        checks['shared_optimizer_shape']=all(np.shape(x)==np.shape(y) for x,y in zip(jax.tree_leaves(out[0]),jax.tree_leaves(optimizer)))
        checks['parameters_changed']=differences(out[0].target,optimizer.target)['max_abs']>0
        # Each lane's retained BN state must equal its CLEAN first pass, including for SAM.
        def clean_bn_forward(model,state,images,key):
            with flax.nn.stateful(state) as updated:
                with flax.nn.stochastic(key):
                    model(images,train=True)
            return updated
        # Match the training transform's lane layout, keeping a separate clean
        # forward with the ORIGINAL weights/state (no optimizer or SAM perturbation).
        # In JAX 0.2.6, vmap with shared convolution weights folds the mapped
        # image axis into the convolution batch; eight serial calls use another shape.
        bn_forward=jax.jit(clean_bn_forward)
        bn_forward_lanes=jax.jit(jax.vmap(clean_bn_forward,
            in_axes=(None,0,0,0),out_axes=0,axis_name='batch'))
        checks['bn_reference_protocol']='clean_forward_same_vmap_layout_v1'
        checks['bn_lanes']=[]
        checks['bn_serial_lanes']=[]  # Preserve the old cross-layout comparison as evidence.
        grouped=shard(batch); lane_keys=keys_for(0,8)
        expected_lanes=synchronize(bn_forward_lanes(optimizer.target,states,grouped['image'],lane_keys))
        checks['bn_reference_finite']=finite(expected_lanes)
        for lane in range(8):
            budget_check()
            state_lane=jax.tree_map(lambda x:x[lane],states)
            serial_expected=synchronize(bn_forward(optimizer.target,state_lane,grouped['image'][lane],lane_keys[lane]))
            expected=jax.tree_map(lambda x:x[lane],expected_lanes)
            actual=jax.tree_map(lambda x:x[lane],out[1])
            checks['bn_lanes'].append(differences(actual,expected))
            checks['bn_serial_lanes'].append(differences(actual,serial_expected))
        checks['bn_updated']=differences(out[1],states)['max_abs']>0
        checks['bn_lane_variation']=max(float(np.max(np.abs(np.asarray(x)-np.asarray(x)[0]))) for x in jax.tree_leaves(out[1]))
        write_json(run/'diagnostics.json',checks)
        check(checks['bn_updated'] and checks['bn_lane_variation']>0,'BN states did not update independently.')
        check(checks['bn_reference_finite'],'Clean BatchNorm reference is non-finite.')
        check(all(d['max_abs']<=2e-5 and d['relative_l2']<=2e-4 for d in checks['bn_lanes']), 'Retained BN differs from clean first pass with matching vmap layout.')
        checks['eight_bn_states']=all(x.shape[0]==8 for x in jax.tree_leaves(out[1]))
        raw_eval=next(iter(ds.get_test()))
        many_eval=synchronize(mapped_eval(optimizer.target,out[1],shard(raw_eval)))
        mean_bn=jax.tree_map(lambda x:jnp.mean(x,axis=0)[None,...],out[1])
        one_eval=jax.jit(jax.vmap(official.eval_step,in_axes=(None,0,0),axis_name='batch'))
        ref_eval=synchronize(one_eval(optimizer.target,mean_bn,shard(raw_eval,1)))
        checks['bn_eval_difference']=differences(jax.tree_map(lambda x:x[0],many_eval),jax.tree_map(lambda x:x[0],ref_eval))
        write_json(run/'diagnostics.json',checks)
        check(checks['eight_replica_finite'] and checks['eight_bn_states'] and checks['shared_optimizer_step'] and checks['shared_optimizer_shape'] and checks['parameters_changed'],
            'Logical-replica update check failed.')
        check(checks['bn_eval_difference']['relative_l2']<=1e-3,'BatchNorm evaluation check failed.')
        del out,many_eval,ref_eval,expected,actual,expected_lanes,serial_expected,bn_forward,bn_forward_lanes
        # Restart timing from untouched initialization; keep the full 200-epoch schedule.
    if phase=='smoke':
        # One complete epoch using the original mapped step; this is functional evidence only.
        initial_parameters=optimizer.target
        epoch_start=time.monotonic(); batch_count=0; losses=[]; memory_samples=[]
        memory_samples.append(dict(step=0,memory_mib=gpu_memory()))
        for batch in train_batches(0):
            budget_check()
            output=synchronize(mapped_step(optimizer,states,shard(batch),keys_for(optimizer.state.step,8)))
            optimizer,states,metrics,lr=output
            check(finite(output),'Non-finite smoke update.')
            batch_count+=1; losses.append(float(np.mean(np.asarray(metrics['train_loss']))))
            if batch_count % 10 == 0:
                memory_samples.append(dict(step=batch_count,memory_mib=gpu_memory()))
        check(batch_count==195,'Smoke epoch batch count mismatch.')
        check(int(optimizer.state.step)==195,'Smoke optimizer step mismatch.')
        check(differences(optimizer.target,initial_parameters)['max_abs']>0,'Smoke training did not update parameters.')
        epoch_seconds=time.monotonic()-epoch_start
        memory_samples.append(dict(step=batch_count,memory_mib=gpu_memory()))
        epoch_row=dict(epoch=1,training_batches=batch_count,mean_loss=float(np.mean(losses)),
                       final_loss=losses[-1],learning_rate=float(np.asarray(lr)),epoch_seconds=epoch_seconds)
        history=[epoch_row]
        save_checkpoint(1,optimizer,states,history)
        ro,rs,re,rh=load_checkpoint()
        check(re==1 and rh==history,'Smoke checkpoint metadata mismatch.')
        check(differences(ro,optimizer)['max_abs']==0 and differences(rs,states)['max_abs']==0,'Smoke checkpoint restoration changed tensors.')
        evaluated=eval_full(optimizer,states)
        restored_eval=eval_full(ro,rs)
        check(abs(evaluated['test_loss']-restored_eval['test_loss'])<=1e-5 and evaluated['test_error_pct']==restored_eval['test_error_pct'], 'Restored evaluation differs.')
        write_json(run/'smoke.json',dict(status='passed',diagnostic_only=True,epochs=1,training_batches=batch_count,
                   training_examples=batch_count*256,evaluation=evaluated,restored_evaluation=restored_eval,
                   checkpoint_restored_at_epochs=[1],parameters_changed=True,finite_losses=True,
                   epoch_runtime_seconds=epoch_seconds,gpu_memory_samples=memory_samples,
                   gpu_memory_peak_mib=max(x['memory_mib'] for x in memory_samples),history=history))
        write_json(run/'status.json',dict(status='complete',diagnostic_only=True,completed_epochs=1,
                   allocated_wall_seconds=prior_seconds+time.monotonic()-started))
        print('One-epoch functional smoke complete; strict preflight not run.',flush=True)
        return
    startup_seconds=time.monotonic()-started
    epoch_limit=1 if phase=='preflight' else EPOCHS[label]
    for epoch in range(completed,epoch_limit):
        budget_check()
        start=time.monotonic(); after_first=None; count=0; step_seconds=[]; tick=start
        for batch in train_batches(epoch):
            budget_check()
            output=synchronize(mapped_step(optimizer,states,shard(batch),keys_for(optimizer.state.step,8)))
            optimizer,states,metrics,lr=output
            check(finite(metrics) and finite(lr),'Non-finite step metrics.')
            now=time.monotonic(); step_seconds.append(now-tick); tick=now
            count+=1
            if count==1:
                after_first=time.monotonic()
        train_seconds=time.monotonic()-start
        steady_epoch_seconds=(time.monotonic()-after_first)*195/194
        check(count==195,'Expected 195 full batches per epoch.')
        check(finite(optimizer) and finite(states),'Non-finite epoch state.')
        eval_start=time.monotonic(); row=eval_full(optimizer,states)
        row.update(epoch=epoch+1,method=label,seed=seed,last_lr=float(np.asarray(lr)),
            train_seconds=train_seconds,eval_seconds=time.monotonic()-eval_start,
            cumulative_train_seconds=sum(x['train_seconds'] for x in history)+train_seconds)
        history.append(row)
        checkpoint_start=time.monotonic(); save_checkpoint(epoch+1,optimizer,states,history)
        checkpoint_seconds=time.monotonic()-checkpoint_start
        write_json(run/'history.json',history); print(json.dumps(row),flush=True)
        if phase=='preflight':
            ro,rs,re,rh=load_checkpoint()
            check(re==1 and rh==history,'Checkpoint metadata round-trip failed.')
            check(differences(ro,optimizer)['max_abs']==0 and differences(rs,states)['max_abs']==0,'Checkpoint state round-trip failed.')
            write_json(run/'preflight.json',dict(status='passed',signature=signature,checkpoint_roundtrip=True,
                estimated_steady_epoch_seconds=steady_epoch_seconds,evaluation_seconds=row['eval_seconds'],
                checkpoint_seconds=checkpoint_seconds,first_step_and_input_seconds=after_first-start,
                startup_and_diagnostics_seconds=startup_seconds,measured_train_seconds=train_seconds,
                steady_step_mean_seconds=float(np.mean(step_seconds[1:])),
                steady_step_p90_seconds=float(np.percentile(step_seconds[1:],90)),step_seconds=step_seconds,
                planning_epoch_seconds=max(train_seconds,195*float(np.percentile(step_seconds[1:],90)))+row['eval_seconds']+checkpoint_seconds,
                note='Actual synchronized epoch, evaluation and checkpoint timings; p90 steps give conservative planning. Startup includes diagnostic compilation.'))
        write_json(run/'status.json',dict(status='running',completed_epochs=epoch+1,resumed=resume,
            allocated_wall_seconds=prior_seconds+time.monotonic()-started))
    write_json(run/'status.json',dict(status='complete',completed_epochs=epoch_limit,resumed=resume,
        allocated_wall_seconds=prior_seconds+time.monotonic()-started,
        allocated_gpu_hours=(prior_seconds+time.monotonic()-started)/3600))

try:
    run_experiment()
except BaseException as error:
    write_json(run/('failure_%d.json'%int(time.time())),dict(type=type(error).__name__,message=str(error),traceback=traceback.format_exc()))
    if active_run or not (run/'manifest.json').exists():
        write_json(run/'status.json',dict(status='failed',allocated_wall_seconds=prior_seconds+time.monotonic()-started))
    raise
'''
ast.parse(WORKER_SOURCE, feature_version=(3, 8))
print('Runner syntax parsed only; GPU behavior (to verify).')

SUPERVISOR_SOURCE = r'''# Linux-only wall-clock budget supervisor. Does not import any ML packages.
import os, sys, json, time, uuid, signal, subprocess, fcntl, ctypes
from pathlib import Path
config_path, phase, method, seed, resume = sys.argv[1:]
C=json.loads(Path(config_path).read_text())
LIMIT=40*3600.0
assert C['budget_limit_hours']==40.0
ledger_path=Path(C['budget_file']); ledger_path.parent.mkdir(parents=True,exist_ok=True)

def atomic(path,value):
    path=Path(path); tmp=path.with_name(path.name+'.tmp')
    with tmp.open('w') as f:
        json.dump(value,f,indent=2,allow_nan=False); f.write('\n'); f.flush(); os.fsync(f.fileno())
    os.replace(str(tmp),str(path))

def interrupt(signum,frame):
    raise KeyboardInterrupt('Supervisor interrupted')

signal.signal(signal.SIGTERM,interrupt)
with (ledger_path.with_suffix('.lock')).open('a+') as lock:
    # Concurrent launches fail; no stale lock-file deletion or automatic budget reset.
    fcntl.flock(lock,fcntl.LOCK_EX|fcntl.LOCK_NB)
    ledger=json.loads(ledger_path.read_text()) if ledger_path.exists() else dict(schema=1,limit_seconds=LIMIT,attempts=[])
    assert ledger['schema']==1 and ledger['limit_seconds']==LIMIT
    assert all(0<=e['charged_seconds']<=e['reserved_seconds'] for e in ledger['attempts'])
    used=sum(e['charged_seconds'] for e in ledger['attempts'])
    assert 0<=used<=LIMIT
    remaining=LIMIT-used
    if remaining<=30:
        raise RuntimeError('Cumulative 40-hour budget exhausted (30-second shutdown reserve).')
    start=time.monotonic(); deadline=start+remaining
    entry=dict(id=uuid.uuid4().hex,session=C['session'],phase=phase,method=method,seed=int(seed),
               started_unix=time.time(),state='reserved',reserved_seconds=remaining,charged_seconds=remaining)
    # Reserve ALL remaining time durably before spawning. A crashed supervisor never refunds it.
    ledger['attempts'].append(entry); atomic(ledger_path,ledger)
    process=None; code=1; budget_stopped=False; confirmed_exit=True
    parent_pid=os.getpid()
    def child_setup():
        os.setsid()
        libc=ctypes.CDLL(None)
        if libc.prctl(1,signal.SIGKILL)!=0 or os.getppid()!=parent_pid:
            os._exit(125)  # Linux parent-death signal: no orphan GPU worker.
    def stop_child():
        if process is None or process.poll() is not None:
            return
        try:
            os.killpg(process.pid,signal.SIGINT)
            process.wait(timeout=min(3,max(0.1,deadline-time.monotonic()-5)))
        except (subprocess.TimeoutExpired,ProcessLookupError):
            if process.poll() is None:
                os.killpg(process.pid,signal.SIGKILL)
                process.wait(timeout=5)
    try:
        env=os.environ.copy()
        env['SAM_STOP_MONOTONIC']=str(deadline-20)
        process=subprocess.Popen([sys.executable,'-u',str(Path(C['session'])/'runner.py'),
                   phase,config_path,method,seed,resume],env=env,preexec_fn=child_setup)
        try:
            code=process.wait(timeout=max(0.1,deadline-time.monotonic()-10))
        except subprocess.TimeoutExpired:
            budget_stopped=True; code=124; stop_child()
        # Cooperative deadline stops may exit before the outer watchdog.
        if code!=0 and time.monotonic()>=deadline-21:
            budget_stopped=True
    except KeyboardInterrupt:
        code=130
    finally:
        # Ignore further interrupts while killing/reaping the child and settling the ledger.
        signal.signal(signal.SIGINT,signal.SIG_IGN); signal.signal(signal.SIGTERM,signal.SIG_IGN)
        try:
            stop_child()
        except BaseException:
            confirmed_exit=False
        elapsed=time.monotonic()-start
        if confirmed_exit and (process is None or process.poll() is not None):
            entry.update(state='budget_stopped' if budget_stopped else 'finished',
                         charged_seconds=min(remaining,elapsed),observed_seconds=elapsed,exit_code=code,
                         finished_unix=time.time())
        else:
            entry.update(state='unresolved',exit_code=code,observed_seconds=elapsed)
            # Keep the entire reservation charged. No further launch is possible.
        atomic(ledger_path,ledger)
        run=Path(C['session'])/phase/(method+'_seed_'+seed)
        run.mkdir(parents=True,exist_ok=True)
        atomic(run/('budget_attempt_'+entry['id']+'.json'),entry)
        if budget_stopped:
            path=run/'status.json'
            status=json.loads(path.read_text()) if path.exists() else {}
            status.update(status='budget_stopped',budget_attempt=entry['id'])
            atomic(path,status)
        print('Cumulative charged GPU hours:',sum(e['charged_seconds'] for e in ledger['attempts'])/3600,
              '/ 40; attempt:',entry['state'],flush=True)
        if elapsed>remaining or not confirmed_exit:
            print('Shutdown/accounting anomaly: budget locked; inspect scheduler accounting.',flush=True)
            code=124
    sys.exit(code)
'''
ast.parse(SUPERVISOR_SOURCE, feature_version=(3, 8))


In [ ]:
def budget_snapshot():
    assert REPLICATION_BUDGET_HOURS==40.0 and CONFIG['budget_limit_hours']==40.0
    if BUDGET_FILE.exists():
        data=json.loads(BUDGET_FILE.read_text())
        assert data['schema']==1 and data['limit_seconds']==40*3600
        assert all(0<=a['charged_seconds']<=a['reserved_seconds'] for a in data['attempts'])
        charged=sum(a['charged_seconds'] for a in data['attempts'])
        assert 0<=charged<=40*3600
    else:
        charged=0.0
    return dict(charged_hours=charged/3600,remaining_hours=40-charged/3600)

def prepare_session():
    assert PYTHON.is_file() and REPO.is_dir(), 'Complete setup or configure the pinned environment.'
    assert SESSION.resolve()!=REPO.resolve() and REPO.resolve() not in SESSION.resolve().parents
    SESSION.mkdir(parents=True,exist_ok=True)
    files=[('config.json',json.dumps(CONFIG,indent=2)+'\n'),('runner.py',WORKER_SOURCE),('supervisor.py',SUPERVISOR_SOURCE)]
    for name,text in files:
        path=SESSION/name
        if path.exists():
            assert path.read_text()==text, 'Session code/config changed: use a new session, keeping the SAME budget ledger.'
        else:
            path.write_text(text)
    return SESSION/'supervisor.py'

def launch(phase,label,seed,resume=False):
    supervisor=prepare_session()
    folder=SESSION/phase/(label+'_seed_'+str(seed))
    if (folder/'status.json').exists() and json.loads((folder/'status.json').read_text()).get('status')=='complete':
        manifest=json.loads((folder/'manifest.json').read_text())
        assert manifest['identity']['config']==CONFIG
        assert manifest['identity']['worker_sha256']==hashlib.sha256(WORKER_SOURCE.encode()).hexdigest()
        print('Already complete; no launch:',folder)
        return
    assert budget_snapshot()['remaining_hours']>30/3600,'Cumulative budget exhausted.'
    environment=os.environ.copy()
    environment['TFDS_DATA_DIR']=str(DATA_DIR)
    environment['PYTHONPATH']=str(REPO.parent)
    environment['PYTHONHASHSEED']=str(seed)
    environment['CUDA_HOME']=str(ENV_PREFIX)
    environment['CUDA_PATH']=str(ENV_PREFIX)
    environment['PATH']=str(ENV_PREFIX/'bin')+':'+environment.get('PATH','')
    environment['LD_LIBRARY_PATH']=str(ENV_PREFIX/'lib')+':'+environment.get('LD_LIBRARY_PATH','')
    environment['XLA_PYTHON_CLIENT_PREALLOCATE']='false'
    environment.setdefault('CUDA_VISIBLE_DEVICES','0')
    environment['XLA_FLAGS']=configured_xla_flags()
    log=SESSION/(phase+'_'+label+'_seed_'+str(seed)+'.log')
    # Regular-file stdout lets the independent supervisor survive notebook disconnection.
    with log.open('a') as stream,log.open('r') as reader:
        reader.seek(0,2)
        process=subprocess.Popen([str(PYTHON),'-u',str(supervisor),str(SESSION/'config.json'),
                    phase,label,str(seed),str(resume).lower()],cwd=str(REPO.parent),env=environment,
                    stdout=stream,stderr=subprocess.STDOUT,start_new_session=True)
        try:
            while process.poll() is None:
                print(reader.read(),end=''); time.sleep(1)
            print(reader.read(),end='')
        except KeyboardInterrupt:
            # Do not kill the supervisor abruptly: it must reap the worker and settle accounting.
            process.send_signal(signal.SIGINT)
            process.wait()
            raise
    print('Persistent budget:',budget_snapshot())
    if process.returncode!=0 and (folder/'diagnostics.json').exists():
        print('Saved diagnostic report: '+str(folder/'diagnostics.json'),flush=True)
        print((folder/'diagnostics.json').read_text(),flush=True)
    assert process.returncode==0,'Worker stopped: inspect '+str(log)+'. Last committed checkpoint remains available.'

if RUN_PREFLIGHT:
    for method in ['SGD-200','SAM-200']:
        launch('preflight',method,0)
else:
    print('Preflight disabled. Its GPU time will count against the SAME cumulative 40-hour ledger.')


In [ ]:
def selected_runs():
    matrix=[(m,s) for s in SEEDS for m in ['SAM-200','SGD-200','SGD-400']]
    selected=matrix if RUN_SELECTION=='all' else [(m,s) for m,s in matrix if m+'_seed_'+str(s)==RUN_SELECTION]
    assert selected, 'RUN_SELECTION must name a planned run, or all.'
    return selected

def preflight_budget():
    reports={}
    for method in ['SGD-200','SAM-200']:
        folder=SESSION/'preflight'/(method+'_seed_0')
        report=json.loads((folder/'preflight.json').read_text())
        status=json.loads((folder/'status.json').read_text())
        manifest=json.loads((folder/'manifest.json').read_text())
        assert report['status']=='passed' and status['status']=='complete'
        assert report['signature']==manifest['signature']
        assert manifest['identity']['config']==CONFIG
        assert manifest['identity']['worker_sha256']==hashlib.sha256(WORKER_SOURCE.encode()).hexdigest()
        reports[method]=report
    costs={}; estimates={}
    epochs_by_method={'SAM-200':200,'SGD-200':200,'SGD-400':400}
    for method,epochs in epochs_by_method.items():
        r=reports['SAM-200' if method.startswith('SAM') else 'SGD-200']
        # Actual startup/diagnostic compilation + synchronized p90 step, full evaluation, and checkpoint I/O.
        startup=r['startup_and_diagnostics_seconds']
        epoch_seconds=r['planning_epoch_seconds']
        costs[method]=1.2*(startup+epochs*epoch_seconds)/3600
        for seed in SEEDS:
            folder=SESSION/'train'/(method+'_seed_'+str(seed))
            status=json.loads((folder/'status.json').read_text()) if (folder/'status.json').exists() else {}
            committed=sorted((folder/'checkpoints').glob('epoch_*'))
            completed=0
            if status.get('status')=='complete':
                completed=epochs
            elif committed:
                meta=json.loads((committed[-1]/'metadata.json').read_text())
                completed=int(meta['completed_epoch'])
                assert 0<=completed<=epochs
            estimates[(method,seed)]=0.0 if completed==epochs else 1.2*(startup+(epochs-completed)*epoch_seconds)/3600
    total=len(SEEDS)*sum(costs.values())
    selected=selected_runs()
    selected_hours=sum(estimates[pair] for pair in selected)
    budget=budget_snapshot()
    print('Actual-preflight estimates per fresh run (20% allowance):',costs)
    print('Nine-run fresh estimate (information only):',round(total,2),'hours')
    print('Selected remaining runs:',selected,'estimated hours:',round(selected_hours,2))
    print('Cumulative charged / remaining hours:',budget['charged_hours'],'/',budget['remaining_hours'])
    print('Estimates are planning aids; the supervisor enforces the remaining wall-time cap.')
    return costs,selected_hours

if all((SESSION/'preflight'/(m+'_seed_0')/'preflight.json').exists() for m in ['SGD-200','SAM-200']):
    costs,estimated_hours=preflight_budget()
else:
    print('No completed preflight for this session. ResNet-18 timings are not WRN-28-10 estimates.')


## 4. Training, cumulative budget and recovery
Enable `RUN_TRAINING` only after passing preflight and reviewing the estimates. Select one named run or `all`. The full experiment stays SAM-200, SGD-200 and SGD-400 across seeds 0, 1 and 2. The selected **remaining** work must fit the **remaining cumulative budget**; the nine-run fresh estimate is information only. Incomplete final comparisons stay incomplete if 40 hours is insufficient; nothing reduces epochs, seeds or model size automatically.

**Hard ceiling: 40 cumulative GPU-worker hours**, including preflight, smoke attempts using this ledger, initialization, compilation, input/data waits, evaluation, checkpoint I/O, failures and resumes. Accounting conservatively charges wall time for one visible GPU, not GPU utilization. Software setup and idle allocated-GPU time outside these workers are not measurable by the notebook: use scheduler accounting for those charges. Keep one authoritative `WORK_ROOT/gpu_budget.json` across sessions. Moving to another host requires transferring that ledger first; do not create/reset it to regain time or run divergent copies. Concurrent use requires the same POSIX-locking filesystem and is rejected.

A separate Linux supervisor holds an exclusive ledger lock, durably reserves remaining time before launch, sets a cooperative stop deadline and terminates the worker before the hard deadline. It settles actual elapsed time only after confirming worker exit. If the supervisor is killed, Linux parent-death handling kills its worker and the full unresolved reservation remains charged: subsequent launches fail closed. Do not manually clear that reservation; reconcile it against scheduler evidence. An uninterruptible OS/driver or storage failure cannot be ruled out by static checks; use the allocation scheduler's wall-time limit as the external backstop **(to verify)**.

To resume, keep `REUSE_SESSION`, choose the interrupted run and enable `RESUME_EXISTING`. The exact code/config/environment must match. Restore the latest committed epoch with optimizer and all eight BN histories; a partial epoch is rerun and remains charged. Two snapshots are retained. Budget stops never pretend to finish an epoch. A checkpoint directory committed before a pointer/status write is recoverable. Stateful augmentation/GPU kernels need not be bitwise deterministic. Pre-existing sessions from an older notebook revision are rejected rather than silently migrated.

Notebook disconnection does not intentionally stop the standalone supervisor; it writes to a regular log file. To interrupt explicitly, interrupt the launch cell and let the supervisor reap its worker and settle accounting. Keep storage persistent. All watchdog, resume and A100 behavior is **(to verify)**; no experiment has been run during authoring.


In [ ]:
if RUN_DIAGNOSTIC_SMOKE:
    assert RUN_SELECTION=='all' and CONFIG['mode']=='smoke'
    for method in ['SGD-200','SAM-200']:
        launch('smoke',method,0)
elif RUN_TRAINING:
    costs,estimated_hours=preflight_budget()
    assert estimated_hours<=budget_snapshot()['remaining_hours']-30/3600, 'Selected remaining work exceeds cumulative budget: select fewer planned runs, without changing the experiment.'
    selected=selected_runs()
    for method,seed in selected:
        launch('train',method,seed,resume=RESUME_EXISTING)
else:
    print('Training disabled. Review preflight cost before enabling it.')


## 5. Results and paper comparison
Enable `RENDER_RESULTS` only to analyze saved runs. This never launches training. It uses pandas/matplotlib/scipy in the notebook kernel; if unavailable, select a kernel with those existing packages (including the pinned environment). It does not install anything.

Use final epoch 200 or 400 only. Incomplete/failed runs are listed separately. Paper-style SGD selects the lower final test error from its two independent durations for each seed; all raw results stay visible. Never select the best intermediate epoch. Student-t 95% intervals describe our three-seed uncertainty; the paper's reference intervals need not use the same formula. The full aggregate is withheld until all nine runs complete.

Figures and CSV/JSON summaries go to this session's `summary/` outside the source checkout. Retain logs, manifests and checkpoints for provenance. Copy only final small summaries for publication when instructed. Do not tune after viewing test results or claim that matching one row establishes the flatness explanation. A failed or inconclusive reproduction is reportable.

The predeclared rule in Section 1 is applied without changing its thresholds after the run. All three paired comparisons remain visible beside the unselected SGD durations. GPU execution, parameter-count equality and actual reproduction outcome are **(to verify)**.

Cumulative accounting lives in `gpu_budget.json` and its per-attempt receipts. Per-run worker wall-time columns are descriptive; they do not replace the authoritative ledger.


In [ ]:
if RENDER_RESULTS:
    import numpy as np
    import pandas as pd
    import matplotlib.pyplot as plt
    from scipy.stats import t
    from IPython.display import display
    records,curves,statuses=[],[],[]
    for seed in SEEDS:
        for method,last_epoch in [('SAM-200',200),('SGD-200',200),('SGD-400',400)]:
            folder=SESSION/'train'/(method+'_seed_'+str(seed))
            status=json.loads((folder/'status.json').read_text()) if (folder/'status.json').exists() else {'status':'not_started'}
            statuses.append(dict(method=method,seed=seed,**status))
            if (folder/'manifest.json').exists():
                manifest=json.loads((folder/'manifest.json').read_text())
                assert manifest['identity']['config']==CONFIG, 'Saved result config differs.'
                assert manifest['identity']['worker_sha256']==hashlib.sha256(WORKER_SOURCE.encode()).hexdigest(), 'Saved result worker differs.'
            rows=json.loads((folder/'history.json').read_text()) if (folder/'history.json').exists() else []
            curves.extend(rows)
            if status['status']=='complete':
                assert len(rows)==last_epoch and [r['epoch'] for r in rows]==list(range(1,last_epoch+1))
                final=dict(rows[-1],allocated_gpu_hours=status['allocated_gpu_hours'],resumed=status['resumed'])
                final['initialization_sha256']=json.loads((folder/'initialization.json').read_text())['parameter_sha256']
                records.append(final)
    display(pd.DataFrame(statuses))
    output=SESSION/'summary'; output.mkdir(parents=True,exist_ok=True)
    pd.DataFrame(statuses).to_csv(output/'run_status.csv',index=False)
    (output/'budget_summary.json').write_text(json.dumps(budget_snapshot(),indent=2))
    print('Authoritative cumulative budget:',budget_snapshot())
    if curves:
        history=pd.DataFrame(curves)
        history.to_csv(output/'epoch_history.csv',index=False)
        fig,ax=plt.subplots(figsize=(9,4))
        colors={'SAM-200':'#dd8452','SGD-200':'#4c72b0','SGD-400':'#55a868'}
        for (method,seed),group in history.groupby(['method','seed']):
            ax.plot(group.epoch,group.test_error_pct,color=colors[method],alpha=0.65,label=method+' seed '+str(seed))
        ax.set(xlabel='Completed epoch',ylabel='Official test error (%)',title='WRN-28-10 / CIFAR-10: recorded runs (including partial runs)')
        ax.legend(ncol=3,fontsize=8); ax.grid(alpha=0.2); fig.tight_layout()
        fig.savefig(output/'test_error_curves.png',dpi=160); plt.show()
    if records:
        final=pd.DataFrame(records)
        assert final.groupby('seed').initialization_sha256.nunique().max()==1, 'Paired methods must share initial weights.'
        if final.seed.nunique()>1:
            assert final.groupby('seed').initialization_sha256.first().nunique()==final.seed.nunique(), 'Different seeds must have different initial weights.'
        final.to_csv(output/'final_per_seed.csv',index=False)
        display(final[['method','seed','test_error_pct','test_accuracy_pct','allocated_gpu_hours','resumed']])
    if len(records)==9:
        raw=final.pivot(index='seed',columns='method',values='test_error_pct')
        raw['SGD paper-style']=raw[['SGD-200','SGD-400']].min(axis=1)
        raw['SGD selected duration']=raw[['SGD-200','SGD-400']].idxmin(axis=1)
        paired_columns={}
        for baseline in ['SGD paper-style','SGD-200','SGD-400']:
            column=baseline+' minus SAM (pp)'
            raw[column]=raw[baseline]-raw['SAM-200']
            paired_columns[baseline]=column
        raw.to_csv(output/'paired_comparison.csv')
        summaries=[]; methods=['SAM-200','SGD-200','SGD-400','SGD paper-style']
        for method in methods:
            values=raw[method].values; mean=values.mean(); sd=values.std(ddof=1)
            ci=float(t.ppf(0.975,len(values)-1)*sd/np.sqrt(len(values)))
            summaries.append(dict(method=method,n=len(values),mean_test_error_pct=mean,sd_pct=sd,ci95_halfwidth_pct=ci))
        summary=pd.DataFrame(summaries)
        summary.to_csv(output/'aggregate.csv',index=False)
        paired_rows=[]
        for baseline,column in paired_columns.items():
            differences=raw[column].to_numpy(dtype=float)
            assert np.isfinite(differences).all()
            count=len(differences); mean=float(differences.mean()); sd=float(differences.std(ddof=1))
            halfwidth=float(t.ppf(0.975,count-1)*sd/np.sqrt(count))
            paired_rows.append(dict(comparison=baseline+' minus SAM',n=count,mean_difference_pp=mean,
                sd_difference_pp=sd,ci95_lower_pp=mean-halfwidth,ci95_upper_pp=mean+halfwidth))
        paired_summary=pd.DataFrame(paired_rows)
        paired_summary.to_csv(output/'paired_difference_summary.csv',index=False)
        display(summary); display(raw); display(paired_summary)
        sam_mean=float(raw['SAM-200'].mean())
        sgd_mean=float(raw['SGD paper-style'].mean())
        primary=paired_rows[0]
        criteria=dict(sam_mean_lower=sam_mean<sgd_mean,
            paired_ci_above_zero=primary['ci95_lower_pp']>0,
            sam_within_0_3_pp=2.4<=sam_mean<=3.0,
            sgd_within_0_3_pp=3.2<=sgd_mean<=3.8)
        assessment=dict(status='reproduced' if all(criteria.values()) else 'criterion not met',
            primary_baseline='SGD paper-style',sam_mean_error_pct=sam_mean,
            sgd_mean_error_pct=sgd_mean,paired_difference=primary,criteria=criteria,
            rule='All nine runs; SAM mean < selected SGD mean; paired 95% CI above zero; means within 0.3 pp of 2.7/3.5 inclusive.')
        (output/'reproduction_assessment.json').write_text(json.dumps(assessment,indent=2))
        print('Predeclared reproduction assessment:',assessment['status'])
        display(pd.DataFrame([criteria]))
        fig,ax=plt.subplots(figsize=(9,4))
        for i,method in enumerate(methods):
            values=raw[method].values
            ax.scatter(i+np.linspace(-0.1,0.1,len(values)),values,color='black',zorder=3)
            row=summaries[i]
            ax.errorbar(i,row['mean_test_error_pct'],yerr=row['ci95_halfwidth_pct'],fmt='o',capsize=5,color='#4c72b0')
        ax.axhspan(2.6,2.8,color='#dd8452',alpha=0.12)
        ax.axhline(2.7,color='#dd8452',ls='--',label='Paper SAM: 2.7 +/- 0.1%')
        ax.axhspan(3.4,3.6,color='#55a868',alpha=0.12)
        ax.axhline(3.5,color='#55a868',ls='--',label='Paper SGD: 3.5 +/- 0.1%')
        ax.set_xticks(range(4)); ax.set_xticklabels(methods)
        ax.set(ylabel='Final test error (%)',title='Three seeds: points, mean and Student-t 95% CI')
        ax.legend(); ax.grid(axis='y',alpha=0.2); fig.tight_layout()
        fig.savefig(output/'paper_comparison.png',dpi=160); plt.show()
        (output/'interpretation_metadata.json').write_text(json.dumps(dict(
            paper=dict(SAM_error_pct=2.7,SGD_error_pct=3.5,ci95_halfwidth_pct=0.1,seeds=5),
            replication_seeds=SEEDS,logical_subbatches=8,physical_gpus=1,
            note='Descriptive replication only. No causal flatness conclusion or selection of intermediate epochs.'),indent=2))
    else:
        (output/'reproduction_assessment.json').write_text(json.dumps(dict(status='not assessed (to verify)',completed_runs=len(records),required_runs=9),indent=2))
        print('Not assessed (to verify):',len(records),'of 9 runs complete. No missing-seed aggregation.')
else:
    print('Results rendering disabled. No results have been fabricated or pre-filled.')
